In [1]:
import torch
import polars as pl

In [2]:
# loading the dataset from hugging face
from datasets import load_dataset
dataset = load_dataset("mohamed-khalil/ATHAR")

In [3]:
#separating the data into training and testing data
train_data=dataset['train'].to_polars()
test_data=dataset['test'].to_polars()

- Note: Stop words won't be removed as we want to test them in the translation

In [4]:
# manually clean the data from links and invalid values through regular expression
# add padding to indicate beginning and end of the words
# tokenize the words
# produce a list of array polars fields
def tokenize_native(t):
  # /s to match any whitespace
  t=t.with_columns(
    pl.all()
    .str.to_lowercase() # convert the english letters to lower case
    .str.replace_all(r'http\S+|www\S+|@|#', '') # strip out the links and special characters
    .str.replace_all(r'^\w\s', ' ') # delete any non alphanumeric word followed by a space
    .str.replace_all(r'\s+', ' ') # delete any consecutive spaces
  ).with_columns(
    pl.format('<sos> {} <eos>', pl.col('arabic')), # add beginning and ending of the sentence from right to left
    pl.format('<sos {} <eos>', pl.col('english')) # add beginning and ending of the sentence from left to right
  ).with_columns(
      pl.all().str.split(' ') # tokenize words
  )
  return t

train_data=tokenize_native(train_data)
test_data=tokenize_native(test_data)

In [5]:
# see what the sentnece is like in both arabic and english
def sen(d, index):
    if isinstance(d['arabic'][index], pl.Series):
        arabic=d['arabic'][index].to_list()
        english=d['english'][index].to_list()
    else:
        arabic=d['arabic'][index].split(' ')
        english=d['english'][index].split(' ')
    for ar, eng in zip(arabic, english):
        print(f"Arabic: {ar}, English: {eng}")

In [6]:
# Pad polars list to the maximum one
# find the maximum list
# Add in text words to ensure all list poems are within the same length
def padding_df(t):
# add padding to the data to ensure they are compatible to be converted to a tensor type
  for col in t.columns:
    max=t.select(pl.col(col)).with_columns(pl.col(col).list.len()).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
        pl.col(col).list.concat(
            pl.lit("<pad>").repeat_by(max-pl.col(col).list.len()) # to ensure compatibleness, repeat by whole size - list len
            # for example if the list len is 3 max is 8 then add only five elements
        ).cast(pl.List(pl.Categorical)).to_physical()) # encode the categorical data to be accepted in torch

  return t

train_data=padding_df(train_data)
test_data=padding_df(test_data)


In [7]:
# cast the values to an integer for easier tensor conversion
def cast_Utf(t):
  for col in t.columns:
    max=t.select(pl.col(col)
    ).with_columns(
        pl.col(col).list.len()
        ).max().item(0,0) # take the maximum length of the list
    t=t.with_columns(
      pl.col(col).cast(pl.Array(pl.UInt32, shape=(max)))
    )

  return t

train_data=cast_Utf(train_data)
test_data=cast_Utf(test_data)

In [ ]:
#TODO: prepare the training engine
#TODO: prepare the inference engine
# code for inference to get back the data to the original format
train_data.with_columns(pl.all().cast(pl.List(pl.Categorical)))

## Data Analysis and Preprocessing
- in this part you are required to to conduct proper analysis of the above data
- You are also required to preprocess the above data in manner where is ready for modelling

## Modelling Section
- In this part you are required to build two models transformer and Attention based sequence to sequence model.

In [8]:
import torch
from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val= train_test_split(train_data['arabic'], train_data['english'], test_size=0.3)
x_train, x_val, y_train, y_val=x_train.to_torch().to(torch.long), x_val.to_torch().to(torch.long), y_train.to_torch().to(torch.long), y_val.to_torch().to(torch.long)

In [9]:
# # make all data parameteric on the type of device available either cuda or cpu
# device='cuda' if torch.cuda.is_available() else 'cpu'
# x_train=x_train.to(device)
# y_train=y_train.to(device)
# x_val=x_val.to(device)
# y_val=y_val.to(device)

## Attention Based Sequence to Sequence Model

In [10]:
import torch.nn as nn

class Encoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, p):
    super(Encoder, self).__init__()
    self.hidden_size=hidden_size
    self.num_layers=num_layers
    self.dropout=nn.Dropout(p)
    self.embedding=nn.Embedding(num_embeddings=input_size, embedding_dim=embedding_size)
    self.lstm=nn.LSTM(embedding_size, hidden_size, batch_first=True, num_layers=self.num_layers) # maybe you don't need to initialize a dropout at that layer

  def forward(self, x):
    # x is [batches, sentences]
    output=self.embedding(x)
    # x is now [batches, sentences, words_embeddings]
    output=self.dropout(output)
    # To prevent overfitting
    output, (hidden, cell)=self.lstm(output)
    # x is [batch, sentences, embeddings]
    return output, hidden, cell

In [11]:
class Decoder(nn.Module):
  def __init__(self, input_size, embedding_size, hidden_size, num_layers, output_size, p):
    super(Decoder, self).__init__()
    self.output_size=output_size
    self.embedding_size=embedding_size
    self.num_layers=num_layers
    self.dropout=nn.Dropout(p)
    # note we are using input size to fed the embedding layer as it represents the index not the actual dimension
    self.embedding=nn.Embedding(num_embeddings=input_size, embedding_dim=embedding_size)
    self.lstm=nn.LSTM(embedding_size, hidden_size, batch_first=True, num_layers=self.num_layers)
    self.out=nn.Linear(hidden_size, output_size)

  def forward(self, input, hidden, cell):
    # Inject the data the the embedding layer and then the dropout layer
      # the input shape given is [batch, sentence]
      embed=self.embedding(input)
      # batch is 1 because its predicting one sentence at a time
      # hidden and cell need refactoring because they are obtained for one sentence so
      embed=self.dropout(embed) # prevent overfitting
      output, (hidden, cell)=self.lstm(embed, (hidden, cell))
      # Use the same stored hidden and cell states to decode the data
      output=self.out(output)
      return output, hidden, cell

In [12]:
import random
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super(Seq2Seq, self).__init__()
        self.encoder=encoder
        self.decoder=decoder

    def forward(self, encoder_input, targets, teacher_forcing_ratio):
        # input shape is [batch, sequence_length, input]
        batch_length=encoder_input.shape[0] # take the total length
        sentence_length=targets.shape[1] # take the length of each sentence
        vocabs_length=self.decoder.output_size
        outputs=torch.zeros(batch_length, sentence_length, vocabs_length)
        encoder_output, hidden, cell= self.encoder(encoder_input)
        decoder_input=targets[:,0].long() # take the first sentence out of the decoder output
        for i in range (1, sentence_length):
            # now we run the decoder
            decoder_output, hidden, cell=self.decoder(decoder_input.unsqueeze(1), hidden, cell) # I think I should change the encoder hidden and cell
            # each loop returns one word at a time
            # now we add this word to the outputs
            outputs[:,i,:]=decoder_output.squeeze(1)
            top_word=decoder_output.squeeze(1).argmax(1)
            teacher_force=random.uniform(0.0, 1.0) < teacher_forcing_ratio 
            # if the teacher_force_ration bigger than the generated random then we take the actual output as the next input, else we take the top generated word as the input
            decoder_input=targets[:, i] if teacher_force else top_word
        
        return outputs

In [13]:
# prepare the data
from torch.utils.data import DataLoader
def prepare_data(data) -> DataLoader:
    return DataLoader(data, batch_size=50, shuffle=False, num_workers=2)

In [14]:
import torch.optim as optim
def prepare_optim(params, lr, decay):
    opt=optim.AdamW(params, lr=lr, weight_decay=decay)
    criterion=nn.CrossEntropyLoss(ignore_index=0)
    return opt, criterion

In [15]:
# prepare the parameters
input_size=x_train.max() + 1 # set the input size based on the largest element for the embedding layer
embedding_size=64 # the dimensions of the embedding layer
hidden_size=512 # the output of the encoder
num_layers=1
dropout=0.5
output_size=y_train.shape[1]
encoder=Encoder(input_size=input_size, embedding_size=embedding_size, hidden_size=hidden_size, num_layers=num_layers, p=dropout)
decoder=Decoder(input_size=input_size, embedding_size=embedding_size, hidden_size=hidden_size, num_layers=num_layers, output_size=output_size, p=dropout)

In [19]:
# Start the training

# prepare the data
train_x=prepare_data(x_train)
train_y=prepare_data(y_train)
val_x=prepare_data(x_val)
val_y=prepare_data(y_val)
#prepare the model
model=Seq2Seq(encoder, decoder)
optim, loss=prepare_optim(model.parameters(), 0.003, 0.001)

## The Method for training follows the Rust burn format

In [20]:
num_epochs=15
for i in range(num_epochs):
    train_loss=0
    val_loss=0
    for x_t,y_t,x_v, y_v in zip(train_x, train_y,val_x, val_y):
        optim.zero_grad()
        output=model(x_t, y_t,0.4)
        print(y_t.shape)
        print(output.shape)
        grad=loss(output, y_t)
        grad.backword()
        optim.step()
        train_loss+=grad
        
        model.eval()
        with model.no_grads():
            output=model(x_v, y_v,0.5)
            grad=loss(output,y_v)
            val_loss+=grad
    print(f"---Train Loss :{train_loss}---")
    print(f"---Validation Loss: {val}---")

torch.Size([50, 203])
torch.Size([50, 203, 203])


IndexError: Target 3672 is out of bounds.